# Week 9: First-Order Logic: Resolution

This notebook follows the Week 9 case study: represent the knowledge base as clauses and prove that John likes peanuts using refutation and unification.

**Source:** `PPTs/Week9.pptx`.


### CNF and refutation
To use resolution, remove implications, move negations inward, standardize variables, Skolemize existential variables, drop universal quantifiers, and distribute OR over AND to obtain clauses. The example below is already supplied in clause form so the program can focus on first-order resolution. Add the negated goal and seek the empty clause.

## Program — First-order logic resolution

**Theory.** Resolution proves a query by contradiction: negate the query, add it to a clause-form knowledge base, then repeatedly resolve complementary literals. Unification supplies substitutions when predicate arguments contain variables. Deriving the empty clause (□) proves the query.

**Objective.** Represent the Week 9 food example as first-order clauses and prove `Likes(John, Peanuts)` by resolution.

**Libraries.** Python standard library only.

**Algorithm.**
1. Enter premise clauses in disjunction form. 2. Add the negated query. 3. Standardize variables apart. 4. Unify complementary literals and add resolvents. 5. Stop at the empty clause or the clause limit.

**Complexity.** The number of resolvents may grow exponentially; the user-set clause limit bounds time and memory.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The direct proof uses the “John likes all food” clause, the fact that Peanuts is food, and the negated goal. Other case-study premises are included in the KB as stated in the slide.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

# Resolution for first-order clauses. Enter each clause as literals separated by |.
import itertools

def split_top(text,separator=','):
    parts=[]; depth=0; start=0
    for i,ch in enumerate(text):
        if ch=='(': depth+=1
        elif ch==')': depth-=1
        elif ch==separator and depth==0: parts.append(text[start:i].strip()); start=i+1
    parts.append(text[start:].strip()); return [p for p in parts if p]
def parse_atom(text):
    text=text.strip(); i=text.find('(')
    if i<1 or not text.endswith(')'): raise ValueError('Atoms must look like Predicate(term, ...).')
    return (text[:i].strip(),tuple(t.strip() for t in split_top(text[i+1:-1])))
def parse_literal(text):
    text=text.strip(); positive=not text.startswith('~')
    return (positive,parse_atom(text if positive else text[1:]))
def parse_clause(text): return tuple(parse_literal(x) for x in split_top(text,'|'))
def isvar(t): return t and t[0].islower()
def apply(t,s):
    while isvar(t) and t in s: t=s[t]
    return t
def unify(a,b,sub):
    if a[0]!=b[0] or len(a[1])!=len(b[1]): return None
    out=dict(sub)
    for x,y in zip(a[1],b[1]):
        x=apply(x,out); y=apply(y,out)
        if x==y: continue
        if isvar(x): out[x]=y
        elif isvar(y): out[y]=x
        else: return None
    return out
def subst_lit(lit,sub): return (lit[0],(lit[1][0],tuple(apply(t,sub) for t in lit[1][1])))
def canonical(clause): return frozenset(clause)
def standardize(clause,index):
    variables=sorted({t for _,a in clause for t in a[1] if isvar(t)})
    mapping={v:f'v{index}_{v}' for v in variables}
    return tuple((sign,(pred,tuple(mapping.get(t,t) for t in args))) for sign,(pred,args) in clause)
def resolve_pair(c1,c2):
    for i,l1 in enumerate(c1):
        for j,l2 in enumerate(c2):
            if l1[0]==l2[0]: continue
            sub=unify(l1[1],l2[1],{})
            if sub is None: continue
            residual=[subst_lit(l,sub) for k,l in enumerate(c1) if k!=i]
            residual += [subst_lit(l,sub) for k,l in enumerate(c2) if k!=j]
            clause=tuple(dict.fromkeys(residual))
            if any((not sign,atom) in clause for sign,atom in clause): continue
            yield clause

def resolve(clauses,max_clauses=500):
    known={canonical(c):c for c in clauses}; support=[clauses[-1]]; steps=[]; serial=0
    while support and len(known)<max_clauses:
        current=support.pop(0)
        for other in list(known.values()):
            if canonical(current)==canonical(other): continue
            serial+=1; left=standardize(current,serial); right=standardize(other,serial+1)
            for result in resolve_pair(left,right):
                key=canonical(result)
                if not result:
                    steps.append((current,other,()))
                    return True,steps,len(known)
                if key not in known:
                    known[key]=result; support.append(result); steps.append((current,other,result))
                    if len(known)>=max_clauses: break
    return False,steps,len(known)
def show_lit(lit):
    sign,(pred,args)=lit
    return ('' if sign else '~')+pred+'('+', '.join(args)+')'
def show_clause(clause): return ' | '.join(show_lit(l) for l in clause) if clause else '□'
def main():
    try: n=int(input('Number of premise clauses: '))
    except ValueError: print('Enter an integer.'); return
    if n<1: print('Enter at least one premise clause.'); return
    try:
        clauses=[parse_clause(input(f'Clause {i+1}: ')) for i in range(n)]
        query=parse_literal(input('Positive atomic query to prove: '))
        limit=int(input('Maximum clauses to derive: '))
    except (ValueError,SyntaxError) as exc: print('Invalid input:',exc); return
    if not query[0]: print('Enter the positive query; its negation is added automatically.'); return
    negated=(False,query[1]); all_clauses=clauses+[ (negated,) ]
    proved,steps,total=resolve(all_clauses,limit)
    print('\nResolution proof steps:')
    for i,(left,right,result) in enumerate(steps,1):
        print(f'{i}. Resolve [{show_clause(left)}] with [{show_clause(right)}] -> {show_clause(result)}')
    print(f'Clauses considered: {total}')
    print('Query proved by refutation.' if proved else 'No refutation found within the clause limit.')


SAMPLE_INPUTS = ['9', '~Food(x) | Likes(John, x)', 'Food(Apple)', 'Food(Vegetables)', '~Eats(x, y) | Killed(x) | Food(y)', 'Eats(Anil, Peanuts)', 'Alive(Anil)', '~Eats(Anil, x) | Eats(Harry, x)', '~Alive(x) | ~Killed(x)', 'Killed(x) | Alive(x)', 'Likes(John, Peanuts)', '500']
with sample_input(SAMPLE_INPUTS):
    main()


Number of premise clauses: 9
Clause 1: ~Food(x) | Likes(John, x)
Clause 2: Food(Apple)
Clause 3: Food(Vegetables)
Clause 4: ~Eats(x, y) | Killed(x) | Food(y)
Clause 5: Eats(Anil, Peanuts)
Clause 6: Alive(Anil)
Clause 7: ~Eats(Anil, x) | Eats(Harry, x)
Clause 8: ~Alive(x) | ~Killed(x)
Clause 9: Killed(x) | Alive(x)
Positive atomic query to prove: Likes(John, Peanuts)
Maximum clauses to derive: 500

Resolution proof steps:
1. Resolve [~Likes(John, Peanuts)] with [~Food(x) | Likes(John, x)] -> ~Food(Peanuts)
2. Resolve [~Food(Peanuts)] with [~Eats(x, y) | Killed(x) | Food(y)] -> ~Eats(v14_x, Peanuts) | Killed(v14_x)
3. Resolve [~Eats(v14_x, Peanuts) | Killed(v14_x)] with [Eats(Anil, Peanuts)] -> Killed(Anil)
4. Resolve [~Eats(v14_x, Peanuts) | Killed(v14_x)] with [~Eats(Anil, x) | Eats(Harry, x)] -> Killed(Harry) | ~Eats(Anil, Peanuts)
5. Resolve [~Eats(v14_x, Peanuts) | Killed(v14_x)] with [~Alive(x) | ~Killed(x)] -> ~Eats(v28_x, Peanuts) | ~Alive(v28_x)
6. Resolve [Killed(Anil)] with [~